In [1]:
# ==========================================
# RAW vs PREPROCESSED side-by-side viewer (Jupyter)
# Fixes mirroring/inversion by forcing RAW to use the SAME Orientation (RAS)
# (and optionally the SAME Spacing) as your preprocessing pipeline.
#
# Also includes an option to disable PRE cropping (body_crop/center_crop) for easier alignment.
# ==========================================

%matplotlib inline

import os, sys, glob
import copy
import yaml
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    Orientationd,
    Spacingd,
)

# -------------------------
# SET THIS
# -------------------------
CONFIG_PATH = "../configs/overfit_one.yaml"  # <-- change if needed


# -------------------------
# find project root (folder containing src/)
# -------------------------
cwd = os.getcwd()
cand = cwd
root_dir = None
for _ in range(12):
    if os.path.isdir(os.path.join(cand, "src")):
        root_dir = cand
        break
    cand = os.path.dirname(cand)
if root_dir is None:
    raise RuntimeError("Could not find project root containing 'src/'. Set root_dir manually.")

if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.data.transforms import get_val_transforms  # your deterministic preprocessing


# -------------------------
# scan pairs (same naming rule as dataset.py)
# -------------------------
def scan_pairs(data_dir: str):
    image_files = sorted(glob.glob(os.path.join(data_dir, "*.img.nii.gz")))
    if not image_files:
        raise FileNotFoundError(f"No '*.img.nii.gz' files found in {data_dir}")

    pairs = []
    for img_path in image_files:
        base = os.path.basename(img_path).replace(".img.nii.gz", "")
        lbl_path = os.path.join(data_dir, f"{base}.label.nii.gz")
        if os.path.exists(lbl_path):
            pairs.append({"id": base, "image": img_path, "label": lbl_path})
    if not pairs:
        raise FileNotFoundError(f"Found images but no paired labels in {data_dir}")
    return pairs


# -------------------------
# basic volume helpers
# -------------------------
def as_np_chwd(x):
    arr = np.asarray(x)
    if arr.ndim != 4:
        raise ValueError(f"Expected [C,H,W,D], got {arr.shape}")
    return arr.astype(np.float32)

def pick_channel(vol_chwd: np.ndarray, c: int):
    c = int(np.clip(c, 0, vol_chwd.shape[0] - 1))
    return vol_chwd[c]  # [H,W,D]

def n_slices(vol_hwd: np.ndarray, axis: int) -> int:
    return [vol_hwd.shape[0], vol_hwd.shape[1], vol_hwd.shape[2]][axis]

def get_slice(vol_hwd: np.ndarray, axis: int, idx: int, rot90: bool = False) -> np.ndarray:
    idx = int(idx)
    if axis == 0:
        sl = vol_hwd[idx, :, :]
    elif axis == 1:
        sl = vol_hwd[:, idx, :]
    else:
        sl = vol_hwd[:, :, idx]
    if rot90:
        sl = np.rot90(sl)
    return sl

def infer_contrast_limits(vol_hwd: np.ndarray, p_lo=1, p_hi=99):
    x = vol_hwd.astype(np.float32)
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return (0.0, 1.0)
    lo, hi = np.percentile(finite, (p_lo, p_hi))
    if hi <= lo + 1e-8:
        lo, hi = float(finite.min()), float(finite.max())
        if hi <= lo + 1e-8:
            return (0.0, 1.0)
    return (float(lo), float(hi))

def norm_for_display(img2d: np.ndarray, lo: float, hi: float):
    x = img2d.astype(np.float32)
    x = np.clip(x, lo, hi)
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)

def rgba_mask(mask2d_bool: np.ndarray, rgb=(0, 1, 0), alpha=0.55):
    h, w = mask2d_bool.shape
    out = np.zeros((h, w, 4), dtype=np.float32)
    out[..., 0] = float(rgb[0])
    out[..., 1] = float(rgb[1])
    out[..., 2] = float(rgb[2])
    out[..., 3] = mask2d_bool.astype(np.float32) * float(alpha)
    return out


# -------------------------
# Load config + resolve paths
# -------------------------
config_abspath = os.path.abspath(CONFIG_PATH)
config_dir = os.path.dirname(config_abspath)

with open(config_abspath, "r") as f:
    config = yaml.safe_load(f)

data_dir = config["data"]["data_dir"]
if not os.path.isabs(data_dir):
    cand1 = os.path.abspath(os.path.join(config_dir, data_dir))
    cand2 = os.path.abspath(os.path.join(root_dir, data_dir))
    if os.path.isdir(cand1):
        data_dir = cand1
    elif os.path.isdir(cand2):
        data_dir = cand2
    else:
        raise FileNotFoundError(f"Cannot resolve data_dir: {config['data']['data_dir']}")

preprocess_cfg = config["data"].get("preprocess", {})
pairs = scan_pairs(data_dir)

print(f"Found {len(pairs)} paired cases in: {data_dir}")
print("Example case id:", pairs[0]["id"])


# -------------------------
# IMPORTANT: RAW transform now matches PRE orientation (RAS) + optional spacing
# -------------------------
pixdim = tuple(preprocess_cfg.get("pixdim", (1.0, 1.0, 1.0)))
RAW_APPLY_SPACING = True  # True = raw also resampled to pixdim, helps alignment

raw_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        *(
            [Spacingd(keys=["image", "label"], pixdim=pixdim, mode=("bilinear", "nearest"))]
            if RAW_APPLY_SPACING
            else []
        ),
        EnsureTyped(keys=["image", "label"], device=None, track_meta=True),
    ]
)

# PRE transform (your full deterministic preprocessing)
# If you get an error about scikit-image, install it: pip install scikit-image
pre_transform_full = get_val_transforms(preprocess_config=preprocess_cfg)

# Optional: PRE without cropping for easier side-by-side alignment
preprocess_cfg_no_crop = copy.deepcopy(preprocess_cfg)
preprocess_cfg_no_crop["body_crop"] = False
preprocess_cfg_no_crop["center_crop"] = False
pre_transform_no_crop = get_val_transforms(preprocess_config=preprocess_cfg_no_crop)


# -------------------------
# State
# -------------------------
STATE = {
    "case_id": None,
    "raw_img": None, "raw_lbl": None,
    "pre_img": None, "pre_lbl": None,
    "raw_meta": None, "pre_meta": None,
}


def load_case(case, use_pre_no_crop: bool):
    out_raw = raw_transform({"image": case["image"], "label": case["label"]})
    out_pre = (pre_transform_no_crop if use_pre_no_crop else pre_transform_full)(
        {"image": case["image"], "label": case["label"]}
    )

    raw_img = as_np_chwd(out_raw["image"])
    raw_lbl = as_np_chwd(out_raw["label"])
    pre_img = as_np_chwd(out_pre["image"])
    pre_lbl = as_np_chwd(out_pre["label"])

    STATE.update(
        {
            "case_id": case["id"],
            "raw_img": raw_img,
            "raw_lbl": raw_lbl,
            "pre_img": pre_img,
            "pre_lbl": pre_lbl,
            "raw_meta": out_raw.get("image_meta_dict", {}),
            "pre_meta": out_pre.get("image_meta_dict", {}),
        }
    )


# -------------------------
# Widgets
# -------------------------
case_dd = widgets.Dropdown(
    options=[(p["id"], i) for i, p in enumerate(pairs)],
    value=0,
    description="Case:",
    layout=widgets.Layout(width="360px"),
)

axis_dd = widgets.Dropdown(
    options=[("Axis 0 (H)", 0), ("Axis 1 (W)", 1), ("Axis 2 (D)", 2)],
    value=2,
    description="Axis:",
    layout=widgets.Layout(width="250px"),
)

rot90_cb = widgets.Checkbox(value=False, description="rot90", indent=False)

use_pre_no_crop_cb = widgets.Checkbox(
    value=True,
    description="PRE: disable crop (align FOV)",
    indent=False,
)

show_overlay_cb = widgets.Checkbox(value=True, description="Overlay label", indent=False)
show_label_only_cb = widgets.Checkbox(value=True, description="Show label-only row", indent=False)

# relative position slider (0..1000) mapped into raw/pre slice indices independently
rel_slider = widgets.IntSlider(
    value=500, min=0, max=1000, step=1,
    description="Pos:",
    continuous_update=True,
    layout=widgets.Layout(width="700px"),
)

step_back = widgets.Button(description="◀", layout=widgets.Layout(width="60px"))
step_fwd  = widgets.Button(description="▶", layout=widgets.Layout(width="60px"))

play = widgets.Play(interval=60, value=500, min=0, max=1000, step=5)
widgets.jslink((play, "value"), (rel_slider, "value"))

pre_ch_dd = widgets.Dropdown(
    options=[("ch 0", 0)],
    value=0,
    description="PRE ch:",
    layout=widgets.Layout(width="200px"),
)

info_out = widgets.Output()
plot_out = widgets.Output()

def refresh_pre_channel_options():
    pre_img = STATE["pre_img"]
    if pre_img is None:
        return
    opts = [(f"ch {i}", i) for i in range(pre_img.shape[0])]
    pre_ch_dd.options = opts
    pre_ch_dd.value = 0

def step(delta):
    rel_slider.value = int(np.clip(rel_slider.value + delta, rel_slider.min, rel_slider.max))

step_back.on_click(lambda b: step(-10))
step_fwd.on_click(lambda b: step(+10))


def render(*args):
    if STATE["raw_img"] is None:
        return

    ax = int(axis_dd.value)
    rot90 = bool(rot90_cb.value)
    rel = int(rel_slider.value) / 1000.0

    raw_img = STATE["raw_img"]
    raw_lbl = STATE["raw_lbl"]
    pre_img = STATE["pre_img"]
    pre_lbl = STATE["pre_lbl"]

    raw_hwd = pick_channel(raw_img, 0)
    pre_c = int(pre_ch_dd.value)
    pre_hwd = pick_channel(pre_img, pre_c)

    raw_lbl_hwd = pick_channel(raw_lbl, 0)
    pre_lbl_hwd = pick_channel(pre_lbl, 0)

    raw_idx = int(np.clip(round(rel * (n_slices(raw_hwd, ax) - 1)), 0, n_slices(raw_hwd, ax) - 1))
    pre_idx = int(np.clip(round(rel * (n_slices(pre_hwd, ax) - 1)), 0, n_slices(pre_hwd, ax) - 1))

    raw_sl = get_slice(raw_hwd, ax, raw_idx, rot90=rot90)
    pre_sl = get_slice(pre_hwd, ax, pre_idx, rot90=rot90)

    raw_m = get_slice((raw_lbl_hwd > 0.5), ax, raw_idx, rot90=rot90)
    pre_m = get_slice((pre_lbl_hwd > 0.5), ax, pre_idx, rot90=rot90)

    raw_lo, raw_hi = infer_contrast_limits(raw_hwd)
    pre_lo, pre_hi = infer_contrast_limits(pre_hwd)

    raw_disp = norm_for_display(raw_sl, raw_lo, raw_hi)
    pre_disp = norm_for_display(pre_sl, pre_lo, pre_hi)

    overlay = bool(show_overlay_cb.value)
    show_label_only = bool(show_label_only_cb.value)

    with plot_out:
        clear_output(wait=True)

        nrows = 2 if show_label_only else 1
        fig, axes = plt.subplots(nrows, 2, figsize=(14, 7 if show_label_only else 4.6))
        if nrows == 1:
            axes = np.array([axes])

        # RAW
        axes[0, 0].imshow(raw_disp, cmap="gray")
        axes[0, 0].set_title(f"RAW (RAS{' + spacing' if RAW_APPLY_SPACING else ''}) | idx={raw_idx}")
        axes[0, 0].axis("off")
        if overlay:
            axes[0, 0].imshow(rgba_mask(raw_m.astype(bool), rgb=(0,1,0), alpha=0.55))

        # PRE
        axes[0, 1].imshow(pre_disp, cmap="gray")
        axes[0, 1].set_title(f"PRE (ch {pre_c}) | idx={pre_idx}")
        axes[0, 1].axis("off")
        if overlay:
            axes[0, 1].imshow(rgba_mask(pre_m.astype(bool), rgb=(0,1,0), alpha=0.55))

        if show_label_only:
            axes[1, 0].imshow(np.zeros_like(raw_disp), cmap="gray")
            axes[1, 0].imshow(rgba_mask(raw_m.astype(bool), rgb=(0,1,0), alpha=1.0))
            axes[1, 0].set_title("RAW label (green)")
            axes[1, 0].axis("off")

            axes[1, 1].imshow(np.zeros_like(pre_disp), cmap="gray")
            axes[1, 1].imshow(rgba_mask(pre_m.astype(bool), rgb=(0,1,0), alpha=1.0))
            axes[1, 1].set_title("PRE label (green)")
            axes[1, 1].axis("off")

        fig.suptitle(
            f"Case {STATE['case_id']} | axis={ax} | rel={rel:.3f} | rot90={rot90} | pre_no_crop={use_pre_no_crop_cb.value}",
            fontsize=12
        )
        plt.tight_layout()
        plt.show()

    with info_out:
        clear_output(wait=True)
        print("----- CASE INFO -----")
        print("Case ID:", STATE["case_id"])
        print("RAW image [C,H,W,D]:", tuple(raw_img.shape), "| label:", tuple(raw_lbl.shape))
        print("PRE image [C,H,W,D]:", tuple(pre_img.shape), "| label:", tuple(pre_lbl.shape))
        fn_raw = STATE["raw_meta"].get("filename_or_obj", "")
        fn_pre = STATE["pre_meta"].get("filename_or_obj", "")
        print("RAW filename:", os.path.basename(str(fn_raw)) if fn_raw else "")
        print("PRE filename:", os.path.basename(str(fn_pre)) if fn_pre else "")
        print("---------------------")


def on_case_change(*args):
    case = pairs[int(case_dd.value)]
    load_case(case, use_pre_no_crop=bool(use_pre_no_crop_cb.value))
    refresh_pre_channel_options()
    render()

def on_pre_mode_change(*args):
    # reload same case with different pre mode
    on_case_change()

case_dd.observe(on_case_change, names="value")
use_pre_no_crop_cb.observe(on_pre_mode_change, names="value")

axis_dd.observe(render, names="value")
rot90_cb.observe(render, names="value")
show_overlay_cb.observe(render, names="value")
show_label_only_cb.observe(render, names="value")
rel_slider.observe(render, names="value")
pre_ch_dd.observe(render, names="value")


# -------------------------
# init
# -------------------------
on_case_change()

controls_top = widgets.HBox([case_dd, axis_dd, pre_ch_dd, rot90_cb])
controls_mid = widgets.HBox([use_pre_no_crop_cb, show_overlay_cb, show_label_only_cb, step_back, step_fwd, play])
display(controls_top)
display(controls_mid)
display(rel_slider)
display(info_out)
display(plot_out)


Found 800 paired cases in: /data/training_data
Example case id: 000f21


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


IntSlider(value=500, description='Pos:', layout=Layout(width='700px'), max=1000)

Output()

Output()